# 03 — Origem: CSV

Le arquivos CSV da zona de **entrada** e grava na camada `coleta`.

Para colocar arquivos na entrada: console do MinIO em <http://localhost:9001>, bucket da zona de
entrada. Ou deixe em `seed/minio-data/` antes do primeiro boot da stack.

In [ ]:
from lakehouse import sessao, ler_arquivo, gravar, perfil, ZONAS

spark = sessao("03-csv")
print("lendo da zona:", ZONAS["entrada"])

In [ ]:
ARQUIVO = "employees.csv"
DESTINO = "coleta.funcionarios"

df = ler_arquivo(spark, ARQUIVO)     # header=True e inferSchema=True por padrao
perfil(df)

## Quando o arquivo nao e bem-comportado

`ler_arquivo` repassa qualquer opcao do leitor CSV do Spark:

```python
df = ler_arquivo(spark, "vendas.csv",
                 sep=";",                 # separador ponto e virgula
                 encoding="ISO-8859-1",   # arquivo que veio do Excel brasileiro
                 dateFormat="dd/MM/yyyy",
                 nullValue="",
                 mode="PERMISSIVE")       # linha ruim vira nulo em vez de derrubar o job
```

**Ler uma pasta inteira** — util para carga particionada por data:

```python
df = ler_arquivo(spark, "vendas/2026/*.csv")
```

**Numero brasileiro** (`1.234,56`) nao e reconhecido por nenhum leitor. Leia como texto e converta:

```python
from pyspark.sql import functions as F
df = ler_arquivo(spark, "vendas.csv", inferSchema=False)
df = df.withColumn("valor",
        F.regexp_replace(F.regexp_replace("valor", r"\.", ""), ",", ".").cast("double"))
```

## Schema explicito

`inferSchema=True` faz o Spark ler o arquivo **duas vezes** e ainda pode errar (CEP virando int e
perdendo o zero a esquerda). Em producao, declare o schema:

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("EmployeeID",   IntegerType(), True),
    StructField("Name",         StringType(),  True),
    StructField("DepartmentID", StringType(),  True),   # texto: preserva zero a esquerda
])

df = ler_arquivo(spark, ARQUIVO, schema=schema, inferSchema=False)
df.printSchema()

In [ ]:
gravar(df, DESTINO, modo="substituir")